# Microsoft Fabric Delta Lake Small-File Health Inspector

**Read-only by default.** Inspect Delta tables in the attached/default Lakehouse using `DESCRIBE DETAIL`. Optionally enumerate active file paths and their sizes (bounded, potentially expensive). No `OPTIMIZE`, `VACUUM`, or table writes are performed.

> Attach a Lakehouse before running. Requires a Fabric Spark notebook with access to its Delta tables. This tool covers tables discoverable in the chosen Spark database, not every Lakehouse in a workspace.

In [ ]:
# Configuration — change these values before running
DATABASE = None                  # None = current Spark database (normally attached Lakehouse)
TABLE_NAME_CONTAINS = ""         # Optional case-insensitive filter
MIN_FILES_TO_FLAG = 100          # Illustrative screening threshold; tune to workload
AVG_MB_TO_FLAG = 32.0           # Heuristic, NOT a universal Fabric recommended file size
MIN_SIZE_MB_TO_FLAG = 512.0     # Avoid flagging tiny tables purely for small averages
DEEP_SCAN = False                # Optional: inspect active file path sizes (higher overhead)
DEEP_SCAN_TABLE_LIMIT = 3       # Limit number of tables eligible for deep inspection
DEEP_SCAN_MAX_FILES = 20000     # Never enumerate more than this many active files per table
SMALL_FILE_THRESHOLD_MB = 32.0 # Deep-scan reporting bucket, not a target file size
SHOW_ALL_TABLES = True

In [ ]:
import json
import math
from datetime import datetime, timezone
from urllib.parse import urlparse, unquote
from pyspark.sql import functions as F
from pyspark.sql import types as T

MB = 1024 ** 2

def quote_identifier(value):
    return "`" + str(value).replace("`", "``") + "`"

def qualified_name(database, table):
    return quote_identifier(database) + "." + quote_identifier(table)

def first_as_dict(df):
    rows = df.take(1)
    return rows[0].asDict(recursive=True) if rows else {}

selected_db = DATABASE or spark.catalog.currentDatabase()
print("Database:", selected_db)
print("Mode:", "metadata + optional deep inspection" if DEEP_SCAN else "metadata-only")

## 1. Inventory Delta tables

Queries the current/selected Spark database. Views and non-Delta tables are skipped. Failed inspections are retained in a separate errors list rather than treated as healthy tables.

In [ ]:
tables = sorted(spark.catalog.listTables(selected_db), key=lambda t: t.name.lower())
records, errors = [], []
for entry in tables:
    if getattr(entry, "isTemporary", False) or entry.tableType.upper() == "VIEW":
        continue
    if TABLE_NAME_CONTAINS.lower() not in entry.name.lower():
        continue
    qname = qualified_name(selected_db, entry.name)
    try:
        d = first_as_dict(spark.sql("DESCRIBE DETAIL " + qname))
        if str(d.get("format", "")).lower() != "delta":
            continue
        files = int(d.get("numFiles") or 0)
        size = int(d.get("sizeInBytes") or 0)
        avg_mb = size / files / MB if files else None
        props = d.get("properties") or {}
        flag = files >= MIN_FILES_TO_FLAG and size >= MIN_SIZE_MB_TO_FLAG * MB and avg_mb is not None and avg_mb < AVG_MB_TO_FLAG
        records.append({
            "database": selected_db, "table": entry.name, "files": files,
            "size_gb": round(size / 1024**3, 3),
            "avg_file_mb": round(avg_mb, 2) if avg_mb is not None else None,
            "partition_columns": ",".join(d.get("partitionColumns") or []),
            "location": d.get("location"),
            "adaptive_target": str(props.get("delta.targetFileSize.adaptive", "not exposed")),
            "review_candidate": flag,
        })
    except Exception as exc:
        errors.append({"table": entry.name, "error": str(exc)[:500]})

print(f"Delta tables inspected: {len(records)}; inspection errors: {len(errors)}")
if records:
    summary_df = spark.createDataFrame(records)
    display(summary_df.orderBy(F.desc("review_candidate"), F.desc("files")))
else:
    print("No accessible Delta tables matched the selected filters.")
if errors:
    display(spark.createDataFrame(errors))

## 2. Optional active-file distribution (expensive)

`DataFrame.inputFiles()` returns the set of paths Spark associates with the Delta snapshot. For each path, Hadoop filesystem metadata is queried to obtain its physical length. This does not read row contents, but can produce heavy metadata traffic and driver memory use. File counts over the configured limit are skipped. Results may differ if concurrent writers commit while the inspection runs. Storage access varies by identity and runtime.

**Disabled by default.** Enable for a few tables only. No `VACUUM` or compaction is performed. If your Spark implementation does not provide reliable `inputFiles()` results, use the official Delta Analyzer for detailed analysis instead.

In [ ]:
deep_results, deep_errors = [], []
if DEEP_SCAN:
    candidates = sorted(records, key=lambda r: (not r["review_candidate"], -r["files"]))[:DEEP_SCAN_TABLE_LIMIT]
    jvm = spark.sparkContext._jvm
    hconf = spark.sparkContext._jsc.hadoopConfiguration()
    for r in candidates:
        name = qualified_name(r["database"], r["table"])
        try:
            paths = spark.table(name).inputFiles()
            if len(paths) != r["files"]:
                deep_errors.append({"table": r["table"], "reason": f"inputFiles count {len(paths)} != DESCRIBE DETAIL numFiles {r['files']}; distribution skipped"})
                continue
            if len(paths) > DEEP_SCAN_MAX_FILES:
                deep_errors.append({"table": r["table"], "reason": "exceeds DEEP_SCAN_MAX_FILES"})
                continue
            sizes = []
            for path in paths:
                jp = jvm.org.apache.hadoop.fs.Path(path)
                fs = jp.getFileSystem(hconf)
                sizes.append(int(fs.getFileStatus(jp).getLen()))
            sizes.sort()
            count = len(sizes)
            threshold = SMALL_FILE_THRESHOLD_MB * MB
            pct = lambda n: round(100 * n / count, 1) if count else None
            deep_results.append({
                "table": r["table"], "active_files": count,
                "under_threshold_files": sum(v < threshold for v in sizes),
                "under_threshold_pct": pct(sum(v < threshold for v in sizes)),
                "min_mb": round(sizes[0] / MB, 2) if count else None,
                "median_mb": round(sizes[count // 2] / MB, 2) if count else None,
                "p90_mb": round(sizes[min(count - 1, math.ceil(count * 0.9) - 1)] / MB, 2) if count else None,
                "max_mb": round(sizes[-1] / MB, 2) if count else None,
            })
        except Exception as exc:
            deep_errors.append({"table": r["table"], "reason": str(exc)[:500]})
    if deep_results:
        display(spark.createDataFrame(deep_results))
    if deep_errors:
        display(spark.createDataFrame(deep_errors))
else:
    print("Deep scan disabled. Set DEEP_SCAN = True to inspect file-size distributions.")

## 3. Next steps (manual, not executed)

- Compare candidates with *observed* query duration, ingest frequency, partition layout, and the current adaptive target.
- Review `DESCRIBE HISTORY` for ingest and recent optimization activity.
- Consider `OPTIMIZE database.table` **only** when permitted and supported for the specific table and writing pattern.
- Avoid automatic `VACUUM` based on this report; retention and concurrent-reader safety need a separate review.
- For Power BI Direct Lake, consider row-group layout and transient transcoding effects of optimization.

The result is a **diagnostic heuristic**, not a capacity-cost estimate or a universal maintenance recommendation.

In [ ]:
print("Completed (UTC):", datetime.now(timezone.utc).isoformat())
print("Total tables:", len(records))
print("Review candidates:", sum(r["review_candidate"] for r in records))
print("Metadata errors:", len(errors), "| Deep-scan errors/skips:", len(deep_errors))
print("No table mutations were performed.")